In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from mpl_toolkits.axes_grid1 import make_axes_locatable
import pandas as pd
from tqdm import trange
from copy import deepcopy

# Datasets

In [ ]:
# Dataset - 1

num_samples = 40
np.random.seed(45)

# Generate data
x1 = np.random.uniform(-20, 20, num_samples)
f_x1 = 100*x1 + 1
eps1 = np.random.randn(num_samples)
y1 = f_x1 + eps1

In [ ]:
# Dataset - 2

np.random.seed(45)

# Generate data
x2 = np.random.uniform(-1, 1, num_samples)
f_x2 = 3*x2 + 4
eps2 = np.random.randn(num_samples)
y2 = f_x2 + eps2

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("Dataset 1", "Dataset 2"))

fig.add_trace(go.Scatter(x=x1, y=y1, mode='markers', name='Dataset 1', marker=dict(color='green')), row=1, col=1)
fig.add_trace(go.Scatter(x=x1, y=f_x1, mode='lines', name='True function 1', line=dict(color='blue')), row=1, col=1)

fig.add_trace(go.Scatter(x=x2, y=y2, mode='markers', name='Dataset 2', marker=dict(color='orange')), row=1, col=2)
fig.add_trace(go.Scatter(x=x2, y=f_x2, mode='lines', name='True function 2', line=dict(color='red')), row=1, col=2)

fig.update_layout(title_text="Visualisation plot of the two datasets")
fig.show()

# Gradient Descent function

In [ ]:
def stochastic_gradient_descent(x, y, batch_size=None, learning_rate=0.001, momentum=0, num_epochs=-1, epsilon=0.1, random_seed=45, verbos=True):
    # sanity checks
    # x and y must have the same length
    assert len(x) == len(y), 'x and y must have the same length'

    # x has to be a numpy array or convertible to a numpy array
    assert isinstance(x, (np.ndarray, list, tuple)), 'x must be a numpy array or convertible to a numpy array'

    # y has to be a numpy array or convertible to a numpy array
    assert isinstance(y, (np.ndarray, list, tuple)), 'y must be a numpy array or convertible to a numpy array'

    # batch_size must be a positive integer or None
    assert (batch_size is None) or (isinstance(batch_size, int) and batch_size > 0), 'batch_size must be a positive integer or None'

    # learning rate must be a positive number
    assert learning_rate > 0, 'learning_rate must be a positive number'

    # momentum must be between 0 and 1
    assert 0 <= momentum <= 1, 'momentum must be between 0 and 1'

    # num_epochs must be a positive integer or -1
    assert isinstance(num_epochs, int) and (num_epochs > 0 or num_epochs == -1), 'num_epochs must be a positive integer or -1'

    # epsilon must be a positive number or None
    assert isinstance(epsilon, (int, float)) and epsilon > 0, 'epsilon must be a positive number'

    # random_seed must be an integer
    assert isinstance(random_seed, int), 'random_seed must be an integer'

    # verbos must be a boolean
    assert isinstance(verbos, bool), 'verbos must be a boolean'

    # convert x to numpy array
    if not isinstance(x, np.ndarray):
        x = np.array(x)
        
    if len(x.shape) == 1:
        x = x.reshape(-1, 1)

    # convert y to numpy array
    if not isinstance(y, np.ndarray):
        y = np.array(y)

    y = y.reshape(-1, 1)

    # set random seed
    np.random.seed(random_seed)
    
    if batch_size is None:
        batch_size = x.shape[0]

    # Initialize weights
    w = np.random.randn(x.shape[1])
    w = w.reshape(-1, 1)
    b = np.random.randn(1)

    w_prev_grad = np.zeros(w.shape)
    b_prev_grad = np.zeros(b.shape)

    ws = []
    bs = []

    ws.append(deepcopy(w))
    bs.append(deepcopy(b))

    # Initialize cost
    epoch_costs = []
    iteration_costs = []

    cost = np.mean(((x @ w + b) - y)**2)
    epoch_costs.append(cost)
    iteration_costs.append(cost)

    epoch = 0
    e_print = 1 if num_epochs == -1 else num_epochs//10

    convergence = num_epochs == -1
    # Gradient descent
    while epoch != num_epochs:
        epoch_cost = 0
        for batch in range(0, x.shape[0], batch_size):
            # get random indices
            indices = np.random.choice(x.shape[0], batch_size)
            x_batch = x[indices]
            y_batch = y[indices]

            # compute predictions
            y_pred = x_batch @ w + b

            # compute gradients
            w_grad = (-2/batch_size) * x_batch.T @ (y_batch - y_pred)
            b_grad = (-2/batch_size) * np.sum(y_batch - y_pred, axis=0)

            # update weights
            change_w = learning_rate * w_grad + momentum * w_prev_grad
            change_b = learning_rate * b_grad + momentum * b_prev_grad

            w -= change_w
            b -= change_b

            w_prev_grad = change_w
            b_prev_grad = change_b

            ws.append(deepcopy(w))
            bs.append(deepcopy(b))

            # compute cost
            iteration_cost = np.mean((y_batch - y_pred)**2)
            epoch_cost += iteration_cost
            iteration_costs.append(iteration_cost)
            
        epoch_costs.append(epoch_cost)
        epoch += 1

        if verbos and (epoch%e_print == 0):
            print(f'Epoch: {epoch}, Cost: {epoch_cost}')

        if convergence and (epoch!=1) and (np.abs(epoch_costs[-2] - epoch_costs[-1]) < epsilon):
            break
    
    return (ws, bs), epoch_costs, iteration_costs

# Average number of steps it takes to converge to an $\epsilon$-neighborhood

In [ ]:
n_epochs = 15
n_runs = 20

## Without Momentum

### Dataset-1

#### Full-batch

In [ ]:
steps_full1 = []
epochs_full1 = []
n_epochs_full1 = []

for i in trange(n_runs):
    _, epochs_costs, iteration_costs = stochastic_gradient_descent(x1, y1, learning_rate=0.001, num_epochs=-1, epsilon=0.001, random_seed=i, verbos=False)
    steps_full1.append(len(iteration_costs))
    n_epochs_full1.append(len(epochs_costs))

    # trim the epochs to n_epochs
    epochs_full1.append(epochs_costs[:n_epochs + 1])

print(f'Average number of steps: {np.mean(steps_full1)}')
print(f'Average number of epochs: {np.mean(n_epochs_full1)}')

In [ ]:
epochs_full1 = np.array(epochs_full1)

epochs_full_mean1 = np.mean(epochs_full1, axis=0)
epochs_full_std1 = np.std(epochs_full1, axis=0)

epochs_full_upper1 = epochs_full_mean1 + epochs_full_std1

epochs_full_lower1 = epochs_full_mean1 - epochs_full_std1
epochs_full_lower1[epochs_full_lower1 < 0] = 0


fig = go.Figure()

fig.add_trace(go.Scatter(x=np.arange(0, n_epochs+1), y=epochs_full_mean1, mode='lines', name='Mean', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=np.arange(0, n_epochs+1), y=epochs_full_upper1, mode='lines', name='Upper Bound', line=dict(color='green'), fill='tonexty'))
fig.add_trace(go.Scatter(x=np.arange(0, n_epochs+1), y=epochs_full_lower1, mode='lines', name='Lower Bound', line=dict(color='green'), fill='tonexty'))

fig.update_layout(title_text=f"Convergence plot of the {n_runs} runs of the gradient descent on Dataset-1", template="plotly_dark", xaxis_title="Epochs", yaxis_title="Cost")
fig.show()

#### Stochastic

In [ ]:
steps_stochastic1 = []
epochs_stochastic1 = []
n_epochs_stochastic1 = []

for i in trange(n_runs):
    _, epochs_costs, iteration_costs = stochastic_gradient_descent(x1, y1, batch_size=1, learning_rate=0.001, num_epochs=-1, epsilon=0.001, random_seed=i, verbos=False)
    steps_stochastic1.append(len(iteration_costs))
    n_epochs_stochastic1.append(len(epochs_costs))

    # trim the epochs to n_epochs
    epochs_stochastic1.append(epochs_costs[:n_epochs + 1])

print(f'Average number of steps: {np.mean(steps_stochastic1)}')
print(f'Average number of epochs: {np.mean(n_epochs_stochastic1)}')

In [ ]:
epochs_stochastic1 = np.array(epochs_stochastic1)

epochs_stochastic_mean1 = np.mean(epochs_stochastic1, axis=0)
epochs_stochastic_std1 = np.std(epochs_stochastic1, axis=0)

epochs_stochastic_upper1 = epochs_stochastic_mean1 + epochs_stochastic_std1

epochs_stochastic_lower1 = epochs_stochastic_mean1 - epochs_stochastic_std1
epochs_stochastic_lower1[epochs_stochastic_lower1 < 0] = 0


fig = go.Figure()

fig.add_trace(go.Scatter(x=np.arange(0, n_epochs+1), y=epochs_stochastic_mean1, mode='lines', name='Mean', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=np.arange(0, n_epochs+1), y=epochs_stochastic_upper1, mode='lines', name='Upper Bound', line=dict(color='green'), fill='tonexty'))
fig.add_trace(go.Scatter(x=np.arange(0, n_epochs+1), y=epochs_stochastic_lower1, mode='lines', name='Lower Bound', line=dict(color='green'), fill='tonexty'))

fig.update_layout(title_text=f"Convergence plot of the {n_runs} runs of the stochastic gradient descent on Dataset-1", xaxis_title="Epochs", yaxis_title="Cost")
fig.show()

### Dataset-2

#### Full-batch

In [ ]:
steps_full2 = []
epochs_full2 = []
n_epochs_full2 = []

for i in trange(n_runs):
    _, epochs_costs, iteration_costs = stochastic_gradient_descent(x2, y2, learning_rate=0.001, num_epochs=-1, epsilon=0.001, random_seed=i, verbos=False)
    steps_full2.append(len(iteration_costs))
    n_epochs_full2.append(len(epochs_costs))

    # trim the epochs to n_epochs
    epochs_costs = epochs_costs[:n_epochs + 1]
    epochs_full2.append(epochs_costs)

print(f'Average number of steps: {np.mean(steps_full2)}')
print(f'Average number of epochs: {np.mean(n_epochs_full2)}')


In [ ]:
epochs_full2 = np.array(epochs_full2)

epochs_full_mean2 = np.mean(epochs_full2, axis=0)
epochs_full_std2 = np.std(epochs_full2, axis=0)

epochs_full_upper2 = epochs_full_mean2 + epochs_full_std2

epochs_full_lower2 = epochs_full_mean2 - epochs_full_std2
epochs_full_lower2[epochs_full_lower2 < 0] = 0


fig = go.Figure()

fig.add_trace(go.Scatter(x=np.arange(0, n_epochs+1), y=epochs_full_mean2, mode='lines', name='Mean', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=np.arange(0, n_epochs+1), y=epochs_full_upper2, mode='lines', name='Upper Bound', line=dict(color='green'), fill='tonexty'))
fig.add_trace(go.Scatter(x=np.arange(0, n_epochs+1), y=epochs_full_lower2, mode='lines', name='Lower Bound', line=dict(color='green'), fill='tonexty'))

fig.update_layout(title_text=f"Convergence plot of the {n_runs} runs of the gradient descent on Dataset-2", template="plotly_dark", xaxis_title="Epochs", yaxis_title="Cost")
fig.show()

#### Stochastic

In [ ]:
steps_stochastic2 = []
epochs_stochastic2 = []
n_epochs_stochastic2 = []

for i in trange(n_runs):
    _, epochs_costs, iteration_costs = stochastic_gradient_descent(x2, y2, batch_size=1, learning_rate=0.001, num_epochs=-1, epsilon=0.001, random_seed=i, verbos=False)
    steps_stochastic2.append(len(iteration_costs))
    n_epochs_stochastic2.append(len(epochs_costs))

    # trim the epochs to n_epochs
    epochs_costs = epochs_costs[:n_epochs + 1]
    epochs_stochastic2.append(epochs_costs)

print(f'Average number of steps: {np.mean(steps_stochastic2)}')
print(f'Average number of epochs: {np.mean(n_epochs_stochastic2)}')

In [ ]:
epochs_stochastic2 = np.array(epochs_stochastic2)

epochs_stochastic_mean2 = np.mean(epochs_stochastic2, axis=0)
epochs_stochastic_std2 = np.std(epochs_stochastic2, axis=0)

epochs_stochastic_upper2 = epochs_stochastic_mean2 + epochs_stochastic_std2

epochs_stochastic_lower2 = epochs_stochastic_mean2 - epochs_stochastic_std2
epochs_stochastic_lower2[epochs_stochastic_lower2 < 0] = 0


fig = go.Figure()

fig.add_trace(go.Scatter(x=np.arange(0, n_epochs+1), y=epochs_stochastic_mean2, mode='lines', name='Mean', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=np.arange(0, n_epochs+1), y=epochs_stochastic_upper2, mode='lines', name='Upper Bound', line=dict(color='green'), fill='tonexty'))
fig.add_trace(go.Scatter(x=np.arange(0, n_epochs+1), y=epochs_stochastic_lower2, mode='lines', name='Lower Bound', line=dict(color='green'), fill='tonexty'))

fig.update_layout(title_text=f"Convergence plot of the {n_runs} runs of the stochastic gradient descent on Dataset-2", template="plotly_dark", xaxis_title="Epochs", yaxis_title="Cost")
fig.show()

## With Momentum

### Dataset-1

#### Full-batch

In [ ]:
steps_full_momentum1 = []
epochs_full_momentum1 = []
n_epochs_full_momentum1 = []

for i in trange(n_runs):
    _, epochs_costs, iteration_costs = stochastic_gradient_descent(x1, y1, learning_rate=0.001, momentum=0.9, num_epochs=-1, epsilon=0.001, random_seed=i, verbos=False)
    steps_full_momentum1.append(len(iteration_costs))
    n_epochs_full_momentum1.append(len(epochs_costs))

    # trim the epochs to n_epochs
    epochs_costs = epochs_costs[:n_epochs + 1]
    epochs_full_momentum1.append(epochs_costs)

print(f'Average number of steps: {np.mean(steps_full_momentum1)}')
print(f'Average number of epochs: {np.mean(n_epochs_full_momentum1)}')

In [ ]:
epochs_full_momentum1 = np.array(epochs_full_momentum1)

epochs_full_momentum_mean1 = np.mean(epochs_full_momentum1, axis=0)
epochs_full_momentum_std1 = np.std(epochs_full_momentum1, axis=0)

epochs_full_momentum_upper1 = epochs_full_momentum_mean1 + epochs_full_momentum_std1

epochs_full_momentum_lower1 = epochs_full_momentum_mean1 - epochs_full_momentum_std1
epochs_full_momentum_lower1[epochs_full_momentum_lower1 < 0] = 0


fig = go.Figure()

fig.add_trace(go.Scatter(x=np.arange(0, n_epochs+1), y=epochs_full_momentum_mean1, mode='lines', name='Mean', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=np.arange(0, n_epochs+1), y=epochs_full_momentum_upper1, mode='lines', name='Upper Bound', line=dict(color='green'), fill='tonexty'))
fig.add_trace(go.Scatter(x=np.arange(0, n_epochs+1), y=epochs_full_momentum_lower1, mode='lines', name='Lower Bound', line=dict(color='green'), fill='tonexty'))

fig.update_layout(title_text=f"Convergence plot of the {n_runs} runs of the gradient descent with momentum on Dataset-1", template="plotly_dark", xaxis_title="Epochs", yaxis_title="Cost")
fig.show()

#### Stochastic

In [ ]:
steps_stochastic_momentum1 = []
epochs_stochastic_momentum1 = []
n_epochs_stochastic_momentum1 = []

for i in trange(n_runs):
    _, epochs_costs, iteration_costs = stochastic_gradient_descent(x1, y1, batch_size=1, learning_rate=0.001, momentum=0.9, num_epochs=-1, epsilon=0.001, random_seed=i, verbos=False)
    steps_stochastic_momentum1.append(len(iteration_costs))
    n_epochs_stochastic_momentum1.append(len(epochs_costs))

    # trim the epochs to n_epochs
    epochs_costs = epochs_costs[:n_epochs + 1]
    epochs_stochastic_momentum1.append(epochs_costs)

print(f'Average number of steps: {np.mean(steps_stochastic_momentum1)}')
print(f'Average number of epochs: {np.mean(n_epochs_stochastic_momentum1)}')

In [ ]:
epochs_stochastic_momentum1 = np.array(epochs_stochastic_momentum1)

epochs_stochastic_momentum_mean1 = np.mean(epochs_stochastic_momentum1, axis=0)
epochs_stochastic_momentum_std1 = np.std(epochs_stochastic_momentum1, axis=0)

epochs_stochastic_momentum_upper1 = epochs_stochastic_momentum_mean1 + epochs_stochastic_momentum_std1

epochs_stochastic_momentum_lower1 = epochs_stochastic_momentum_mean1 - epochs_stochastic_momentum_std1
epochs_stochastic_momentum_lower1[epochs_stochastic_momentum_lower1 < 0] = 0


fig = go.Figure()

fig.add_trace(go.Scatter(x=np.arange(0, n_epochs+1), y=epochs_stochastic_momentum_mean1, mode='lines', name='Mean', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=np.arange(0, n_epochs+1), y=epochs_stochastic_momentum_upper1, mode='lines', name='Upper Bound', line=dict(color='green'), fill='tonexty'))
fig.add_trace(go.Scatter(x=np.arange(0, n_epochs+1), y=epochs_stochastic_momentum_lower1, mode='lines', name='Lower Bound', line=dict(color='green'), fill='tonexty'))

fig.update_layout(title_text=f"Convergence plot of the {n_runs} runs of the stochastic gradient descent with momentum on Dataset-1", template="plotly_dark", xaxis_title="Epochs", yaxis_title="Cost")
fig.show()

### Dataset-2

#### Full-batch

In [ ]:
steps_full_momentum2 = []
epochs_full_momentum2 = []
n_epochs_full_momentum2 = []

for i in trange(n_runs):
    _, epochs_costs, iteration_costs = stochastic_gradient_descent(x2, y2, learning_rate=0.001, momentum=0.9, num_epochs=-1, epsilon=0.001, random_seed=i, verbos=False)
    steps_full_momentum2.append(len(iteration_costs))
    n_epochs_full_momentum2.append(len(epochs_costs))

    # trim the epochs to n_epochs
    epochs_costs = epochs_costs[:n_epochs + 1]
    epochs_full_momentum2.append(epochs_costs)

print(f'Average number of steps: {np.mean(steps_full_momentum2)}')
print(f'Average number of epochs: {np.mean(n_epochs_full_momentum2)}')

In [ ]:
epochs_full_momentum2 = np.array(epochs_full_momentum2)

epochs_full_momentum_mean2 = np.mean(epochs_full_momentum2, axis=0)
epochs_full_momentum_std2 = np.std(epochs_full_momentum2, axis=0)

epochs_full_momentum_upper2 = epochs_full_momentum_mean2 + epochs_full_momentum_std2

epochs_full_momentum_lower2 = epochs_full_momentum_mean2 - epochs_full_momentum_std2
epochs_full_momentum_lower2[epochs_full_momentum_lower2 < 0] = 0


fig = go.Figure()

fig.add_trace(go.Scatter(x=np.arange(0, n_epochs+1), y=epochs_full_momentum_mean2, mode='lines', name='Mean', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=np.arange(0, n_epochs+1), y=epochs_full_momentum_upper2, mode='lines', name='Upper Bound', line=dict(color='green'), fill='tonexty'))
fig.add_trace(go.Scatter(x=np.arange(0, n_epochs+1), y=epochs_full_momentum_lower2, mode='lines', name='Lower Bound', line=dict(color='green'), fill='tonexty'))

fig.update_layout(title_text=f"Convergence plot of the {n_runs} runs of the gradient descent with momentum on Dataset-2", template="plotly_dark", xaxis_title="Epochs", yaxis_title="Cost")
fig.show()

#### Stochastic

In [ ]:
steps_stochastic_momentum2 = []
epochs_stochastic_momentum2 = []
n_epochs_stochastic_momentum2 = []

for i in trange(n_runs):
    _, epochs_costs, iteration_costs = stochastic_gradient_descent(x2, y2, batch_size=1, learning_rate=0.001, momentum=0.9, num_epochs=-1, epsilon=0.001, random_seed=i, verbos=False)
    steps_stochastic_momentum2.append(len(iteration_costs))
    n_epochs_stochastic_momentum2.append(len(epochs_costs))

    # trim the epochs to n_epochs
    epochs_costs = epochs_costs[:n_epochs + 1]
    epochs_stochastic_momentum2.append(epochs_costs)

print(f'Average number of steps: {np.mean(steps_stochastic_momentum2)}')
print(f'Average number of epochs: {np.mean(n_epochs_stochastic_momentum2)}')

In [ ]:
epochs_stochastic_momentum2 = np.array(epochs_stochastic_momentum2)

epochs_stochastic_momentum_mean2 = np.mean(epochs_stochastic_momentum2, axis=0)
epochs_stochastic_momentum_std2 = np.std(epochs_stochastic_momentum2, axis=0)

epochs_stochastic_momentum_upper2 = epochs_stochastic_momentum_mean2 + epochs_stochastic_momentum_std2

epochs_stochastic_momentum_lower2 = epochs_stochastic_momentum_mean2 - epochs_stochastic_momentum_std2
epochs_stochastic_momentum_lower2[epochs_stochastic_momentum_lower2 < 0] = 0


fig = go.Figure()

fig.add_trace(go.Scatter(x=np.arange(0, n_epochs+1), y=epochs_stochastic_momentum_mean2, mode='lines', name='Mean', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=np.arange(0, n_epochs+1), y=epochs_stochastic_momentum_upper2, mode='lines', name='Upper Bound', line=dict(color='green'), fill='tonexty'))
fig.add_trace(go.Scatter(x=np.arange(0, n_epochs+1), y=epochs_stochastic_momentum_lower2, mode='lines', name='Lower Bound', line=dict(color='green'), fill='tonexty'))

fig.update_layout(title_text=f"Convergence plot of the {n_runs} runs of the stochastic gradient descent with momentum on Dataset-2", template="plotly_dark", xaxis_title="Epochs", yaxis_title="Cost")
fig.show()

# Single Example Convergence

## Some functions to visualize the convergence

In [ ]:
def create_XYZ(x, y, w, b):
    # Generate data
    w1 = np.linspace(w.min(), w.max(), 250)
    b1 = np.linspace(b.min(), b.max(), 250)
    W, B = np.meshgrid(w1, b1)

    # Evaluate the function
    Z = np.zeros((W.shape[0], W.shape[1]))
    for i in range(W.shape[0]):
        for j in range(W.shape[1]):
            Z[i, j] = np.mean((y - W[i, j]*x - B[i, j])**2)

    return W, B, Z


def f1(x1,y1,X,Y):
    sum = 0
    for i in range(len(x1)):
        sum += -2*x1[i]*(y1[i] - Y - X*x1[i])
    return sum/len(x1)


def f2(x1,y1,X,Y):
    sum = 0
    for i in range(len(x1)):
        sum += -2*(y1[i] - Y - X*x1[i])
    return sum/len(x1)
    

def create_contour(X, Y, Z, ax, alpha, scatter_pts=None, filled=True, levels=10, mark_levels=False):
    if filled:
        scatter_color='white'
        contour = ax.contourf(X, Y, Z, levels=levels, alpha=alpha)
    else:
        scatter_color='black'
        contour = ax.contour(X, Y, Z, levels=levels,  alpha=alpha)
    if scatter_pts is not None:
        ax.scatter(scatter_pts[0], scatter_pts[1], s=10, c=scatter_color)
    ax.set_xlabel('w')
    ax.set_ylabel('b')
    ax.set_title('Contour Plot')
    
    # Add a colorbar in between the subplots
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    cbar = plt.colorbar(contour, cax=cax)
    return ax, contour


def plot_surface_and_contour(x, y, w, b, uv = None, stride=4, alpha=1, scatter_pts=None, filled=True, levels=10, scale=100000):
    X, Y, Z = create_XYZ(x, y, w, b)

    # Create the single figure
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))

    # Create the contour plot
    ax, contour = create_contour(X, Y, Z, ax, alpha, scatter_pts, filled, levels)

    uv = [f1,f2]
    
    if uv is not None:
        u = uv[0](x, y, X, Y)
        v = uv[1](x, y, X, Y)
        # Quiver plot for gradient
        ax.quiver(X[::stride, ::stride], Y[::stride, ::stride], u[::stride, ::stride],
                   v[::stride, ::stride], scale=scale)
        # for c in contour, set alpha
        for c in contour.collections:
            c.set_alpha(0.5)
    
    plt.tight_layout(pad=1.0, w_pad=1.0)
    plt.savefig('contour_plot.png')
    plt.show()

## Without Momentum

### Dataset-1

#### Gradient Vectors on Contours

In [ ]:
wm = 97.5
wM = 102.5

bm = -10
bM = 10

N = 10000

ww = np.linspace(wm, wM, N)
bb = np.linspace(bm, bM, N)

In [ ]:
plot_surface_and_contour(x1, y1, ww, bb, scale=10000)

#### Full-batch

In [ ]:
(ws, bs), _, iteration_losses = stochastic_gradient_descent(x1, y1, learning_rate=0.001, num_epochs=-1, epsilon=0.001, random_seed=45, verbos=False)

In [ ]:
# plot the loss
fig = go.Figure()

fig.add_trace(go.Scatter(x=np.arange(0, len(iteration_losses)), y=iteration_losses, mode='lines', name='Loss', line=dict(color='blue')))
fig.update_layout(title_text="Loss of the gradient descent on Dataset-1", xaxis_title="Epochs", yaxis_title="Cost")
fig.show()

In [ ]:
ws_new = np.array(ws[20:]).ravel()
bs_new = np.array(bs[20:]).ravel()

wr = ws_new.max() - ws_new.min()
br = bs_new.max() - bs_new.min()

wm = ws_new.min() - 0.1*wr
wM = ws_new.max() + 0.1*wr

bm = bs_new.min() - 0.1*br
bM = bs_new.max() + 0.1*br

jumps = 1
N = ws_new.shape[0]

ww = np.linspace(wm, wM, N)
bb = np.linspace(bm, bM, N)
WW, BB = np.meshgrid(ww, bb)

Z = np.zeros((N, N))

for i in range(N):
    for j in range(N):
        Z[i, j] = np.mean((y1 - WW[i, j]*x1 - BB[i, j])**2)

In [ ]:
fig = go.Figure(data=[go.Contour(z=Z, x=ww, y=bb, colorscale="Viridis", opacity=0.7, contours=dict(showlabels=True)),
                      # hoverover info: iteration number, iteration loss
                      go.Scatter(x=ws_new, y=bs_new, mode="lines", line=dict(width=2, color="blue"), name="Gradient Descent")
                     ],
                     layout=go.Layout(xaxis=dict(range=[wm, wM], autorange=False, zeroline=False, title="Weights"),
                                      yaxis=dict(range=[bm, bM], autorange=False, zeroline=False, title="Bias"),
                                      title_text="Weights updates of the gradient descent on Dataset-1", hovermode="closest"))

fig.show()

In [ ]:
ws_new = np.array(ws).ravel()
bs_new = np.array(bs).ravel()

wr = ws_new.max() - ws_new.min()
br = bs_new.max() - bs_new.min()

wm = ws_new.min() - 0.1*wr
wM = ws_new.max() + 0.1*wr

bm = bs_new.min() - 0.1*br
bM = bs_new.max() + 0.1*br

N = ws_new.shape[0]

ww = np.linspace(wm, wM, N)
bb = np.linspace(bm, bM, N)
WW, BB = np.meshgrid(ww, bb)

Z = np.zeros((N, N))

for i in range(N):
    for j in range(N):
        Z[i, j] = np.mean((y1 - WW[i, j]*x1 - BB[i, j])**2)

In [ ]:
fig = go.Figure(data=[go.Contour(z=Z, x=ww, y=bb, colorscale="Viridis", opacity=0.7, contours=dict(showlabels=True)),
                      # hoverover info: iteration number, iteration loss
                      go.Scatter(x=ws_new, y=bs_new, mode="lines", line=dict(width=2, color="blue"), name="Gradient Descent")
                     ],
                     layout=go.Layout(xaxis=dict(range=[wm, wM], autorange=False, zeroline=False, title="Weights"),
                                      yaxis=dict(range=[bm, bM], autorange=False, zeroline=False, title="Bias"),
                                      title_text="Weights updates of the gradient descent on Dataset-1", hovermode="closest"))

fig.show()

#### Stochastic

In [ ]:
(ws, bs), _, iteration_losses = stochastic_gradient_descent(x1, y1, batch_size=1, learning_rate=0.001, num_epochs=-1, epsilon=0.001, random_seed=45, verbos=False)

In [ ]:
# plot the loss using matplotlib
plt.plot(iteration_losses)
plt.title("Loss of the stochastic gradient descent on Dataset-1")
plt.xlabel("Epochs")
plt.ylabel("Cost")
plt.show()

In [ ]:
ws_new = np.array(ws).ravel()
bs_new = np.array(bs).ravel()

wr = ws_new.max() - ws_new.min()
br = bs_new.max() - bs_new.min()

wm = ws_new.min() - 0.1*wr
wM = ws_new.max() + 0.1*wr

bm = bs_new.min() - 0.1*br
bM = bs_new.max() + 0.1*br

jumps = 1
N = 100

ww = np.linspace(wm, wM, N)
bb = np.linspace(bm, bM, N)
WW, BB = np.meshgrid(ww, bb)

Z = np.zeros((N, N))

for i in range(N):
    for j in range(N):
        Z[i, j] = np.mean((y1 - WW[i, j]*x1 - BB[i, j])**2)

In [ ]:
# plot the weights updates using matplotlib with contour plot
plt.contourf(WW, BB, Z, 20, alpha=0.7, cmap='viridis')
plt.colorbar()
plt.plot(ws_new, bs_new, color='blue')
plt.title("Weights updates of the stochastic gradient descent on Dataset-1")
plt.xlabel("Weights")
plt.ylabel("Bias")
plt.show()

In [ ]:
ws_new = np.array(ws[25:]).ravel()
bs_new = np.array(bs[25:]).ravel()

wr = ws_new.max() - ws_new.min()
br = bs_new.max() - bs_new.min()

wm = ws_new.min() - 0.1*wr
wM = ws_new.max() + 0.1*wr

bm = bs_new.min() - 0.1*br
bM = bs_new.max() + 0.1*br

jumps = 1
N = 100

ww = np.linspace(wm, wM, N)
bb = np.linspace(bm, bM, N)
WW, BB = np.meshgrid(ww, bb)

Z = np.zeros((N, N))

for i in range(N):
    for j in range(N):
        Z[i, j] = np.mean((y1 - WW[i, j]*x1 - BB[i, j])**2)

In [ ]:
# plot the weights updates using matplotlib with contour plot
plt.contourf(WW, BB, Z, 20, alpha=0.7, cmap='viridis')
plt.colorbar()
plt.plot(ws_new, bs_new, color='blue')
plt.title("Weights updates of the stochastic gradient descent on Dataset-1")
plt.xlabel("Weights")
plt.ylabel("Bias")
plt.show()

### Dataset-2

#### Gradient Vectors on Contours

In [ ]:
wm = 1
wM = 4

bm = 2
bM = 5

N = 10000

ww = np.linspace(wm, wM, N)
bb = np.linspace(bm, bM, N)

In [ ]:
plot_surface_and_contour(x2, y2, ww, bb, scale=100)

#### Full-batch

In [ ]:
(ws, bs), _, iteration_losses = stochastic_gradient_descent(x2, y2, learning_rate=0.001, num_epochs=-1, epsilon=0.001, random_seed=45, verbos=False)

In [ ]:
# plot the loss
fig = go.Figure()

fig.add_trace(go.Scatter(x=np.arange(0, len(iteration_losses)), y=iteration_losses, mode='lines', name='Loss', line=dict(color='blue')))
fig.update_layout(title_text="Loss of the gradient descent on Dataset-2", xaxis_title="Epochs", yaxis_title="Cost")
fig.show()

In [ ]:
ws_new = np.array(ws).ravel()
bs_new = np.array(bs).ravel()

wr = ws_new.max() - ws_new.min()
br = bs_new.max() - bs_new.min()

wm = ws_new.min() - 0.1*wr
wM = ws_new.max() + 0.1*wr

bm = bs_new.min() - 0.1*br
bM = bs_new.max() + 0.1*br

N = ws_new.shape[0]

ww = np.linspace(wm, wM, N)
bb = np.linspace(bm, bM, N)
WW, BB = np.meshgrid(ww, bb)

Z = np.zeros((N, N))

for i in range(N):
    for j in range(N):
        Z[i, j] = np.mean((y2 - WW[i, j]*x2 - BB[i, j])**2)

In [ ]:
fig = go.Figure(data=[go.Contour(z=Z, x=ww, y=bb, colorscale="Viridis", opacity=0.7, contours=dict(showlabels=True)),
                      # hoverover info: iteration number, iteration loss
                      go.Scatter(x=ws_new, y=bs_new, mode="lines", line=dict(width=2, color="blue"), name="Gradient Descent")
                     ],
                     layout=go.Layout(xaxis=dict(range=[wm, wM], autorange=False, zeroline=False, title="Weights"),
                                      yaxis=dict(range=[bm, bM], autorange=False, zeroline=False, title="Bias"),
                                      title_text="Weights updates of the gradient descent on Dataset-1", hovermode="closest"))

fig.show()

#### Stochastic

In [ ]:
(ws, bs), _, iteration_losses = stochastic_gradient_descent(x2, y2, batch_size=1, learning_rate=0.001, num_epochs=-1, epsilon=0.001, random_seed=45, verbos=False)

In [ ]:
# plot the loss using matplotlib
plt.plot(iteration_losses)
plt.title("Loss of the stochastic gradient descent on Dataset-2")
plt.xlabel("Epochs")
plt.ylabel("Cost")
plt.show()

In [ ]:
ws_new = np.array(ws).ravel()
bs_new = np.array(bs).ravel()

wr = ws_new.max() - ws_new.min()
br = bs_new.max() - bs_new.min()

wm = ws_new.min() - 0.1*wr
wM = ws_new.max() + 0.1*wr

bm = bs_new.min() - 0.1*br
bM = bs_new.max() + 0.1*br

jumps = 1
N = 100

ww = np.linspace(wm, wM, N)
bb = np.linspace(bm, bM, N)
WW, BB = np.meshgrid(ww, bb)

Z = np.zeros((N, N))

for i in range(N):
    for j in range(N):
        Z[i, j] = np.mean((y1 - WW[i, j]*x1 - BB[i, j])**2)

In [ ]:
# plot the weights updates using matplotlib with contour plot
plt.contourf(WW, BB, Z, 20, alpha=0.7, cmap='viridis')
plt.colorbar()
plt.plot(ws_new, bs_new, color='blue')
plt.title("Weights updates of the stochastic gradient descent on Dataset-1")
plt.xlabel("Weights")
plt.ylabel("Bias")
plt.show()

In [ ]:
ws_new = np.array(ws[-20000:]).ravel()
bs_new = np.array(bs[-20000:]).ravel()

wr = ws_new.max() - ws_new.min()
br = bs_new.max() - bs_new.min()

wm = ws_new.min() - 0.1*wr
wM = ws_new.max() + 0.1*wr

bm = bs_new.min() - 0.1*br
bM = bs_new.max() + 0.1*br

jumps = 1
N = 100

ww = np.linspace(wm, wM, N)
bb = np.linspace(bm, bM, N)
WW, BB = np.meshgrid(ww, bb)

Z = np.zeros((N, N))

for i in range(N):
    for j in range(N):
        Z[i, j] = np.mean((y1 - WW[i, j]*x1 - BB[i, j])**2)

In [ ]:
# plot the weights updates using matplotlib with contour plot
plt.contourf(WW, BB, Z, 20, alpha=0.7, cmap='viridis')
plt.colorbar()
plt.plot(ws_new, bs_new, color='blue')
plt.title("Weights updates of the stochastic gradient descent on Dataset-1")
plt.xlabel("Weights")
plt.ylabel("Bias")
plt.show()

## With Momentum

### Dataset-1

#### Full-batch

In [ ]:
(ws, bs), _, iteration_costs = stochastic_gradient_descent(x1, y1, learning_rate=0.001, momentum=0.9, num_epochs=-1, epsilon=0.001, random_seed=45, verbos=False)

In [ ]:
# plot the loss
fig = go.Figure()

fig.add_trace(go.Scatter(x=np.arange(0, len(iteration_losses)), y=iteration_losses, mode='lines', name='Loss', line=dict(color='blue')))
fig.update_layout(title_text="Loss of the gradient descent on Dataset-1", xaxis_title="Epochs", yaxis_title="Cost")
fig.show()

In [ ]:
ws_new = np.array(ws).ravel()
bs_new = np.array(bs).ravel()

wr = ws_new.max() - ws_new.min()
br = bs_new.max() - bs_new.min()

wm = ws_new.min() - 0.1*wr
wM = ws_new.max() + 0.1*wr

bm = bs_new.min() - 0.1*br
bM = bs_new.max() + 0.1*br

N = ws_new.shape[0]

ww = np.linspace(wm, wM, N)
bb = np.linspace(bm, bM, N)
WW, BB = np.meshgrid(ww, bb)

Z = np.zeros((N, N))

for i in range(N):
    for j in range(N):
        Z[i, j] = np.mean((y1 - WW[i, j]*x1 - BB[i, j])**2)

In [ ]:
fig = go.Figure(data=[go.Contour(z=Z, x=ww, y=bb, colorscale="Viridis", opacity=0.7, contours=dict(showlabels=True)),
                      # hoverover info: iteration number, iteration loss
                      go.Scatter(x=ws_new, y=bs_new, mode="lines", line=dict(width=2, color="blue"), name="Gradient Descent")
                     ],
                     layout=go.Layout(xaxis=dict(range=[wm, wM], autorange=False, zeroline=False, title="Weights"),
                                      yaxis=dict(range=[bm, bM], autorange=False, zeroline=False, title="Bias"),
                                      title_text="Weights updates of the gradient descent on Dataset-1", hovermode="closest"))

fig.show()

#### Stochastic

In [ ]:
(ws, bs), _, iteration_costs = stochastic_gradient_descent(x1, y1, batch_size=1, learning_rate=0.001, momentum=0.9, num_epochs=-1, epsilon=0.001, random_seed=45, verbos=False)

In [ ]:
# plot the loss using matplotlib
plt.plot(iteration_losses)
plt.title("Loss of the stochastic gradient descent on Dataset-1")
plt.xlabel("Epochs")
plt.ylabel("Cost")
plt.show()

In [ ]:
ws_new = np.array(ws).ravel()
bs_new = np.array(bs).ravel()

wr = ws_new.max() - ws_new.min()
br = bs_new.max() - bs_new.min()

wm = ws_new.min() - 0.1*wr
wM = ws_new.max() + 0.1*wr

bm = bs_new.min() - 0.1*br
bM = bs_new.max() + 0.1*br

jumps = 1
N = 100

ww = np.linspace(wm, wM, N)
bb = np.linspace(bm, bM, N)
WW, BB = np.meshgrid(ww, bb)

Z = np.zeros((N, N))

for i in range(N):
    for j in range(N):
        Z[i, j] = np.mean((y1 - WW[i, j]*x1 - BB[i, j])**2)

In [ ]:
# plot the weights updates using matplotlib with contour plot
plt.contourf(WW, BB, Z, 20, alpha=0.7, cmap='viridis')
plt.colorbar()
plt.plot(ws_new, bs_new, color='blue')
plt.title("Weights updates of the stochastic gradient descent on Dataset-1")
plt.xlabel("Weights")
plt.ylabel("Bias")
plt.show()

### Dataset-2

#### Full-batch

In [ ]:
(ws, bs), _, iteration_losses = stochastic_gradient_descent(x2, y2, learning_rate=0.001, momentum=0.9, num_epochs=-1, epsilon=0.001, random_seed=45, verbos=False)

In [ ]:
# plot the loss
fig = go.Figure()

fig.add_trace(go.Scatter(x=np.arange(0, len(iteration_losses)), y=iteration_losses, mode='lines', name='Loss', line=dict(color='blue')))
fig.update_layout(title_text="Loss of the gradient descent on Dataset-2", xaxis_title="Epochs", yaxis_title="Cost")
fig.show()

In [ ]:
ws_new = np.array(ws).ravel()
bs_new = np.array(bs).ravel()

wr = ws_new.max() - ws_new.min()
br = bs_new.max() - bs_new.min()

wm = ws_new.min() - 0.1*wr
wM = ws_new.max() + 0.1*wr

bm = bs_new.min() - 0.1*br
bM = bs_new.max() + 0.1*br

N = ws_new.shape[0]

ww = np.linspace(wm, wM, N)
bb = np.linspace(bm, bM, N)
WW, BB = np.meshgrid(ww, bb)

Z = np.zeros((N, N))

for i in range(N):
    for j in range(N):
        Z[i, j] = np.mean((y2 - WW[i, j]*x2 - BB[i, j])**2)

In [ ]:
fig = go.Figure(data=[go.Contour(z=Z, x=ww, y=bb, colorscale="Viridis", opacity=0.7, contours=dict(showlabels=True)),
                      # hoverover info: iteration number, iteration loss
                      go.Scatter(x=ws_new, y=bs_new, mode="lines", line=dict(width=2, color="blue"), name="Gradient Descent")
                     ],
                     layout=go.Layout(xaxis=dict(range=[wm, wM], autorange=False, zeroline=False, title="Weights"),
                                      yaxis=dict(range=[bm, bM], autorange=False, zeroline=False, title="Bias"),
                                      title_text="Weights updates of the gradient descent on Dataset-2", hovermode="closest"))

fig.show()

#### Stochastic

In [ ]:
(ws, bs), _, iteration_losses = stochastic_gradient_descent(x2, y2, batch_size=1, learning_rate=0.001, momentum=0.9, num_epochs=-1, epsilon=0.001, random_seed=45, verbos=False)

In [ ]:
# plot the loss using matplotlib
plt.plot(iteration_losses)
plt.title("Loss of the stochastic gradient descent on Dataset-2")
plt.xlabel("Epochs")
plt.ylabel("Cost")
plt.show()

In [ ]:
ws_new = np.array(ws).ravel()
bs_new = np.array(bs).ravel()

wr = ws_new.max() - ws_new.min()
br = bs_new.max() - bs_new.min()

wm = ws_new.min() - 0.1*wr
wM = ws_new.max() + 0.1*wr

bm = bs_new.min() - 0.1*br
bM = bs_new.max() + 0.1*br

jumps = 1
N = 100

ww = np.linspace(wm, wM, N)
bb = np.linspace(bm, bM, N)
WW, BB = np.meshgrid(ww, bb)

Z = np.zeros((N, N))

for i in range(N):
    for j in range(N):
        Z[i, j] = np.mean((y1 - WW[i, j]*x1 - BB[i, j])**2)

In [ ]:
# plot the weights updates using matplotlib with contour plot
plt.contourf(WW, BB, Z, 20, alpha=0.7, cmap='viridis')
plt.colorbar()
plt.plot(ws_new, bs_new, color='blue')
plt.title("Weights updates of the stochastic gradient descent on Dataset-2")
plt.xlabel("Weights")
plt.ylabel("Bias")
plt.show()